In [1]:
!pip install -q transformers datasets accelerate evaluate rouge-score
!pip install -q pymupdf pdfplumber Pillow

import os
import torch
import fitz 
from PIL import Image
import io
from datasets import load_dataset, DatasetDict
from transformers import (
    NougatProcessor, 
    VisionEncoderDecoderModel, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments,
    default_data_collator
)
import evaluate
device = "cuda" if torch.cuda.is_available() else "cpu"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 103.1 MB/s eta 0:00:00


In [2]:
!apt-get update -y
!apt-get install tesseract-ocr -y
!pip install -q pytesseract

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu

In [3]:
import os
from datasets import load_dataset, DatasetDict
dataset = load_dataset("marcodsn/arxiv-markdown", split="train[:500]")
def check_local_pdf(example):
    pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-big/arxiv_dataset_pdfs/", f"{example['arxiv_id']}.pdf")
    return os.path.exists(pdf_path)
dataset = dataset.filter(check_local_pdf)

total_found = len(dataset)
if total_found == 0:
    raise FileNotFoundError("404 Not Found.")

test_size = 0.15 
valid_size = 0.15
train_test_split = dataset.train_test_split(test_size=test_size, seed=42)
test_dataset = train_test_split['test']
train_valid_split = train_test_split['train'].train_test_split(test_size=valid_size, seed=42)
train_dataset = train_valid_split['train']
valid_dataset = train_valid_split['test']
final_datasets = DatasetDict({
    'train': train_dataset,
    'valid': valid_dataset,
    'test': test_dataset
})

print(f"Train: {len(final_datasets['train'])} mẫu")
print(f"Valid: {len(final_datasets['valid'])} mẫu")
print(f"Test: {len(final_datasets['test'])} mẫu")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Train: 361 mẫu
Valid: 64 mẫu
Test: 75 mẫu


In [4]:
import pytesseract
import fitz
from PIL import Image
import io
import evaluate
import os
from tqdm.auto import tqdm

rouge = evaluate.load("rouge")
baseline_preds = []
ground_truths = []
content_col = 'markdown' if 'markdown' in final_datasets['test'].column_names else 'content'
for example in tqdm(final_datasets['test']):
    arxiv_id = example['arxiv_id']
    ground_truth = str(example[content_col])[:1024] 
    pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-big/arxiv_dataset_pdfs/", f"{arxiv_id}.pdf")
    
    if not os.path.exists(pdf_path):
        continue
        
    try:
        doc = fitz.open(pdf_path)
        page = doc.load_page(0) 
        pix = page.get_pixmap(dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        doc.close()
        pred_text = pytesseract.image_to_string(img)
        baseline_preds.append(pred_text)
        ground_truths.append(ground_truth)
        
    except Exception as e:
        print(f"Bỏ qua mẫu {arxiv_id} do lỗi: {e}")
baseline_results = rouge.compute(predictions=baseline_preds, references=ground_truths, use_stemmer=True)
for key, value in baseline_results.items():
    print(f"- {key}: {round(value, 4)}")

  0%|          | 0/75 [00:00<?, ?it/s]

- rouge1: 0.496
- rouge2: 0.4541
- rougeL: 0.4731
- rougeLsum: 0.4848


In [5]:
import os
import io
from PIL import Image
import fitz 
from transformers import NougatProcessor
processor = NougatProcessor.from_pretrained("facebook/nougat-small", use_fast=False)

available_columns = final_datasets['train'].column_names
content_col = 'markdown' if 'markdown' in available_columns else 'content'

def preprocess_function(examples):
    pixel_values, labels = [], []
    batch_arxiv_ids = examples['arxiv_id']
    batch_markdowns = examples[content_col]
    
    for arxiv_id, markdown_text in zip(batch_arxiv_ids, batch_markdowns):
        pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-big/arxiv_dataset_pdfs/", f"{arxiv_id}.pdf")
        if not os.path.exists(pdf_path):
            print(f"Bỏ qua mẫu {arxiv_id} vì file không tồn tại cục bộ.")
            continue
            
        try:
            doc = fitz.open(pdf_path)
            page = doc.load_page(0) 
            pix = page.get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            doc.close()
            processed_img = processor(img, return_tensors="pt").pixel_values.squeeze()
            pixel_values.append(processed_img)
            tokenized_text = processor.tokenizer(
                str(markdown_text)[:1024], padding="max_length", 
                max_length=512, truncation=True, return_tensors="pt"
            ).input_ids.squeeze()
            tokenized_text[tokenized_text == processor.tokenizer.pad_token_id] = -100
            labels.append(tokenized_text)
            
        except Exception as e:
            print(f"Bỏ qua mẫu {arxiv_id} do lỗi đọc/phân tích file: {e}")
            continue
            
    return {"pixel_values": pixel_values, "labels": labels}
train_dataset = final_datasets['train'].map(preprocess_function, batched=True, remove_columns=available_columns)
valid_dataset = final_datasets['valid'].map(preprocess_function, batched=True, remove_columns=available_columns)
test_dataset = final_datasets['test'].map(preprocess_function, batched=True, remove_columns=available_columns)

preprocessor_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Map:   0%|          | 0/361 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

In [6]:
rouge = evaluate.load("rouge")
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_preds = processor.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return result

In [7]:
model = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-small")
start_token = processor.tokenizer.bos_token_id
if start_token is None:
    start_token = processor.tokenizer.convert_tokens_to_ids("<s>")
model.config.decoder_start_token_id = start_token
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder.decoder_start_token_id = start_token
model.config.decoder.pad_token_id = processor.tokenizer.pad_token_id
model.config.use_cache = False
training_args = Seq2SeqTrainingArguments(
    output_dir="./ocr_finetuned_checkpoints",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=3e-5,
    num_train_epochs=2,
    eval_strategy="steps",           
    eval_steps=100,
    save_steps=100,
    logging_steps=20,
    predict_with_generate=True,
    report_to="none"
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

In [8]:
from transformers import Seq2SeqTrainer, default_data_collator
model.config.use_cache = False
if hasattr(model.config, 'decoder'):
    model.config.decoder.use_cache = False
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=processor.tokenizer,
    data_collator=default_data_collator,   
    compute_metrics=compute_metrics,
)
trainer.train()
model.config.use_cache = True
if hasattr(model.config, 'decoder'):
    model.config.decoder.use_cache = True
trainer.save_model("./best_ocr_model")
processor.save_pretrained("./best_ocr_model")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
100,0.674531,0.337018,0.068018,0.059847,0.067486,0.067763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['./best_ocr_model/processor_config.json']

In [9]:
trainer.model.config.use_cache = False
if hasattr(trainer.model.config, 'decoder'):
    trainer.model.config.decoder.use_cache = False
test_results = trainer.predict(test_dataset)

print("Test metric")
for key, value in test_results.metrics.items():
    print(f"- {key}: {value}")
trainer.model.config.use_cache = True
if hasattr(trainer.model.config, 'decoder'):
    trainer.model.config.decoder.use_cache = True

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test metric
- test_loss: 0.35286957025527954
- test_rouge1: 0.11435779266609428
- test_rouge2: 0.101928864992296
- test_rougeL: 0.11414581900989913
- test_rougeLsum: 0.11406428804180498
- test_runtime: 98.0248
- test_samples_per_second: 0.765
- test_steps_per_second: 0.194


In [10]:
from transformers import VisionEncoderDecoderModel, NougatProcessor
import fitz
from PIL import Image
import io
import re

def my_finetuned_ocr_tool_full(pdf_path):
    my_model = VisionEncoderDecoderModel.from_pretrained("./best_ocr_model").to(device)
    my_processor = NougatProcessor.from_pretrained("./best_ocr_model", use_fast=False)
    
    full_markdown_result = []
    
    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)
        for page_num in range(total_pages):
            print(f"Page {page_num + 1}/{total_pages}...")
            page = doc.load_page(page_num)
            pix = page.get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            
            pixel_values = my_processor(img, return_tensors="pt").pixel_values.to(device)
            
            outputs = my_model.generate(
                pixel_values,
                max_new_tokens=1024, 
                bad_words_ids=[[my_processor.tokenizer.unk_token_id]],
            )
            
            sequence = my_processor.batch_decode(outputs, skip_special_tokens=True)[0]
            clean_sequence = sequence.replace("[wgn]", "").replace("[MISSING_PAGE_EMPTY]", "").strip()
            clean_sequence = re.sub(r'\n\s*\n', '\n\n', clean_sequence)
            
            full_markdown_result.append(f"\n{clean_sequence}")
    return "\n\n---\n\n".join(full_markdown_result)

test_pdf_path = "/kaggle/input/datasets/danghiennguyen/test-pdf/Prokct.pdf"
result_md = my_finetuned_ocr_tool_full(test_pdf_path)
print(result_md)

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Page 1/8...
Page 2/8...
Page 3/8...
Page 4/8...
Page 5/8...
Page 6/8...
Page 7/8...
Page 8/8...

## Final Assignment
## I. Project requirements

## 1. Project Objective

The objective of this project is to design, implement, and document an end-to-end NLP system that solves a realistic industry or business problem.

This project must reflect real-world NLP practice, including problem formulation, data handling, model development, deployment considerations, and ethical responsibility.

Important: This is not a research-only or notebook-only project. Your system must be designed as if it were going into production.

## 2. Business Problem Definition

- Requirements: You must begin with a clear business use case where NLP provides measurable value.

- Deliverables: Submit a Problem Definition Document (1-2 pages) containing:

* Business context and motivation
* Target users or stakeholders
* Description of the problem being solved
* Explanation of why NLP is required
* Success metrics, in